# Lab 04: Tracing & Reproducibility

## Business Context

The CTO asks: *"How do we know the scorer is consistent? Can you prove why it gave Coinbase a 43?"* Enable tracing so every tool call, LLM invocation, and retrieval step is logged. Then tag runs so that rubric version changes are reproducible and auditable.

**After this lab:** Every agent invocation produces a trace — a nested tree of spans showing each LLM call, tool call, and retrieval step with timing. Runs are tagged with rubric version, endpoint, and chunking parameters so you can reproduce any result and compare across rubric versions.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Application Development (30%) + Evaluation & Monitoring (12%) |
| **Key Skills** | MLflow autologging, traces vs spans, experiment tagging, `search_runs`, `search_traces`, version comparison |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | ~$1-2 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph unitycatalog-ai[databricks] unitycatalog-langchain databricks-agents --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
LLM_ENDPOINT = "databricks-llama-4-maverick"

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print(f"LLM      : {LLM_ENDPOINT}")

from shared.lab_utils import build_agent
agent, tools, llm = build_agent(include_scoring=True)
print(f"Agent loaded with {len(tools)} tools: {[t.name for t in tools]}")

## A. Enable MLflow Tracing

**One call** instruments the entire LangChain/LangGraph stack — LLM calls, tool calls, retrieval steps, and prompt rendering all produce spans automatically.

Key concepts:
- **Trace** — the complete record of a single agent invocation (root span + all child spans)
- **Span** — a single unit of work within a trace (e.g., one LLM call, one tool call, one retrieval query)
- **Autologging** — `mlflow.langchain.autolog()` patches LangChain internals so every call creates spans without any extra code

The experiment path organises traces and runs in the MLflow UI under a shared folder.

In [ ]:
import mlflow

# One call instruments everything: LLM calls, tool calls, retrieval
mlflow.langchain.autolog(log_traces=True)

username = spark.sql("SELECT current_user()").first()[0]
experiment_path = f"/Users/{username}/ipo-analyzer/lab-04-tracing"
mlflow.set_experiment(experiment_path)

print(f"Tracing enabled.")
print(f"Experiment: {experiment_path}")

In [ ]:
# Query 1: Research — exercises search_filings + LLM synthesis
q1 = "What does Snowflake say about its competitive advantages in its S-1?"
print(f"Q: {q1}")
print("=" * 70)
r1 = agent.invoke({"messages": [{"role": "user", "content": q1}]})
print(r1["messages"][-1].content)

In [ ]:
# Query 2: Stock lookup — exercises get_stock_performance
q2 = "What was Coinbase's 12-month stock return after its IPO?"
print(f"Q: {q2}")
print("=" * 70)
r2 = agent.invoke({"messages": [{"role": "user", "content": q2}]})
print(r2["messages"][-1].content)

In [ ]:
# Query 3: Clarity scoring — exercises score_clarity
q3 = "Score Coinbase's risk factors section for messaging clarity."
print(f"Q: {q3}")
print("=" * 70)
r3 = agent.invoke({"messages": [{"role": "user", "content": q3}]})
print(r3["messages"][-1].content)

In [ ]:
# Inspect the traces that were auto-captured
experiment = mlflow.get_experiment_by_name(experiment_path)
traces = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=10,
)

print(f"Captured {len(traces)} traces")
print()

for _, trace in traces.iterrows():
    print(f"  Trace ID : {trace['request_id']}")
    print(f"  Status   : {trace['status']}")
    print(f"  Duration : {trace.get('execution_time_ms', 'N/A')} ms")
    print()

## B. Tag Runs for Reproducibility

MLflow **tags** and **params** make runs reproducible:
- **Tags** — mutable key-value metadata (rubric version, environment, requester). Can be updated after the run.
- **Params** — immutable key-value configuration logged once at run start (chunk_size, model name). Cannot be changed after logging.
- **Metrics** — numeric values that can be logged at each step (e.g., answer length, score value).

Tagging with `rubric_version` lets us answer the CTO's question: *"Which version of the rubric produced that Coinbase score?"*

In [ ]:
scoring_query = "Score Coinbase's risk factors section for messaging clarity."

with mlflow.start_run(run_name="coinbase-clarity-v1") as run:
    # Tags: mutable metadata — rubric version, environment, endpoint
    mlflow.set_tags({
        "rubric_version": "v1",
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": "1000",
        "chunk_overlap": "200",
        "query_type": "clarity_scoring",
    })

    # Params: immutable configuration — logged once
    mlflow.log_params({
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "num_retrieval_results": 5,
    })

    # Run the scoring query
    result_v1 = agent.invoke({"messages": [{"role": "user", "content": scoring_query}]})
    answer_v1 = result_v1["messages"][-1].content

    # Log a metric: answer length as a proxy for response verbosity
    mlflow.log_metric("answer_length_chars", len(answer_v1))

    run_id_v1 = run.info.run_id
    print(f"Run ID (v1): {run_id_v1}")
    print(f"Answer length: {len(answer_v1)} chars")
    print()
    print(answer_v1)

## C. Compare Rubric Versions

The team proposes a new rubric (v2) that uses smaller chunks (256 chars) to give the scorer more granular context. We simulate this as a tag change and run the same query. Then `mlflow.search_runs()` lets us compare both versions side by side.

> **Tags vs Params for version control:** Tags are the right choice here because they can be updated *after* a run if the rubric is retroactively renamed. Params are immutable and better for configuration values that define *how* the code ran.

In [ ]:
# Simulate v2 rubric: smaller chunk_size for finer-grained context
with mlflow.start_run(run_name="coinbase-clarity-v2") as run:
    mlflow.set_tags({
        "rubric_version": "v2",
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": "256",       # Changed from 1000
        "chunk_overlap": "50",     # Changed from 200
        "query_type": "clarity_scoring",
    })

    mlflow.log_params({
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": 256,
        "chunk_overlap": 50,
        "num_retrieval_results": 5,
    })

    # Same query, different rubric context
    result_v2 = agent.invoke({"messages": [{"role": "user", "content": scoring_query}]})
    answer_v2 = result_v2["messages"][-1].content

    mlflow.log_metric("answer_length_chars", len(answer_v2))

    run_id_v2 = run.info.run_id
    print(f"Run ID (v2): {run_id_v2}")
    print(f"Answer length: {len(answer_v2)} chars")
    print()
    print(answer_v2)

In [ ]:
import pandas as pd

# Retrieve both versions with a single filter string
experiment = mlflow.get_experiment_by_name(experiment_path)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.rubric_version IN ('v1', 'v2')",
    order_by=["tags.rubric_version ASC"],
)

# Build a side-by-side comparison table
comparison_cols = [
    "tags.rubric_version",
    "tags.chunk_size",
    "tags.chunk_overlap",
    "tags.llm_endpoint",
    "metrics.answer_length_chars",
    "run_id",
]

comparison = runs_df[comparison_cols].copy()
comparison.columns = ["rubric_version", "chunk_size", "chunk_overlap", "llm_endpoint", "answer_length_chars", "run_id"]

print("=== Rubric Version Comparison ===")
display(comparison)

## Before / After

**Before this lab:** When the CTO asks "Why did Coinbase score 43?", the only answer is "we ran the agent and it said so." There is no record of which rubric version was used, what chunks were retrieved, how long each step took, or whether the result would be the same if you ran it again.

**After this lab:** Every invocation produces a trace — a tree of spans showing the exact LLM call, the tool call, and the retrieval step with timing. Every run is tagged with rubric version, chunk size, and endpoint. `search_runs()` makes the comparison auditable.

In [ ]:
# BEFORE: No tracing — black box invocation
print("=== BEFORE: No tracing (black box) ===")
print("agent.invoke({'messages': [...]})")
print("-> Got answer. But: which chunks were retrieved? How long did the LLM call take?")
print("   What rubric version? Can we reproduce this? NO.")
print()

# AFTER: Trace tree for one query (inspect via mlflow.search_traces)
print("=== AFTER: Trace tree for the v1 scoring query ===")
experiment = mlflow.get_experiment_by_name(experiment_path)
traces = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=5,
)

if not traces.empty:
    # Show the most recent trace's structure
    latest = traces.iloc[0]
    print(f"Trace ID  : {latest['request_id']}")
    print(f"Status    : {latest['status']}")
    print(f"Duration  : {latest.get('execution_time_ms', 'N/A')} ms")
    print()
    print("Span hierarchy (LLM call -> tool call -> retrieval):")
    print("  [root] ReAct agent")
    print("    [span] LLM: choose tool          <-- reasoning step")
    print("    [span] Tool: search_filings       <-- retrieval call")
    print("      [span] VectorSearch.similarity_search")
    print("    [span] Tool: score_clarity        <-- scoring call")
    print("    [span] LLM: synthesize answer     <-- final response")
    print()
    print("All steps logged. Duration, inputs, and outputs captured per span.")
else:
    print("No traces found — ensure mlflow.langchain.autolog(log_traces=True) ran before queries.")

In [ ]:
from shared.lab_utils import get_scorecard
get_scorecard()

---

## Exam Preparation

### Key Concepts

| Concept | Definition |
|---|---|
| **Trace** | The complete record of a single agent invocation — a root span plus all nested child spans |
| **Span** | A single unit of work within a trace (one LLM call, one tool call, one retrieval query) with timing, inputs, and outputs |
| **Autologging** | `mlflow.langchain.autolog(log_traces=True)` patches LangChain internals to create spans automatically without extra code |
| **Experiment tags** | Mutable key-value metadata attached to a run (`rubric_version`, `llm_endpoint`); can be updated after the run completes |
| **Params** | Immutable key-value configuration logged once at run start (`chunk_size`, `chunk_overlap`); cannot be changed after logging |
| **`search_runs()`** | MLflow API to query runs by filter string (e.g., tag values, metric ranges); returns a Pandas DataFrame |
| **`search_traces()`** | MLflow API to query traces captured for an experiment; returns a DataFrame with request ID, status, and duration |

### Practice Questions

**Q1.** Which single call enables tracing for all LangChain and LangGraph invocations?

- A) `mlflow.start_run(log_traces=True)`
- B) `mlflow.langchain.autolog(log_traces=True)`
- C) `mlflow.enable_tracing(framework="langchain")`
- D) `agent.enable_tracing(mlflow_client)`

**Answer: B** — `mlflow.langchain.autolog(log_traces=True)` is the single call that instruments LangChain and LangGraph automatically. It must be called before any agent invocations.

---

**Q2.** What is the difference between a trace and a span?

- A) A trace is a single LLM call; a span is the complete agent invocation
- B) A trace is the complete record of one agent invocation; a span is a single unit of work within that trace
- C) Traces are for LLM calls; spans are for tool calls
- D) There is no difference — the terms are interchangeable

**Answer: B** — A trace is the full tree for one invocation (root span + all nested spans). A span is one node in that tree — one LLM call, one retrieval, one tool execution.

---

**Q3.** The CTO wants to update the `rubric_version` tag on old runs after a renaming decision. Can she?

- A) No — tags are immutable once a run is complete
- B) Yes — tags are mutable and can be updated after a run completes
- C) No — only params can be updated post-run
- D) Yes, but only within 24 hours of run completion

**Answer: B** — Tags are mutable. `mlflow.set_tag(run_id, key, value)` can update a tag on any completed run. Params, by contrast, are immutable — they cannot be changed after logging.

---

**Q4.** Which `filter_string` retrieves runs tagged with either `v1` or `v2` rubric?

- A) `"tags.rubric_version = 'v1' OR tags.rubric_version = 'v2'"`
- B) `"tags.rubric_version IN ('v1', 'v2')"`
- C) `"params.rubric_version IN ('v1', 'v2')"`
- D) `"rubric_version IN ['v1', 'v2']"`

**Answer: B** — The correct syntax is `tags.<key> IN ('val1', 'val2')`. Tags use the `tags.` prefix in filter strings. Option A is also valid SQL-style syntax that MLflow accepts, but B is the canonical form.

---

**Q5.** Why are tags preferred over params for `rubric_version`?

- A) Tags are indexed faster than params in MLflow
- B) Tags are mutable — they can be updated if the rubric is retroactively renamed or versioned
- C) Params are not visible in `search_runs()` filter strings
- D) Tags support richer data types (lists, dicts) than params

**Answer: B** — `rubric_version` may need to change after the fact (e.g., a rubric gets a new canonical name). Tags are mutable so they can be updated on existing runs. Params are immutable — they are the right choice for values that must never change (like the exact chunk_size used at run time).

### Cost Breakdown

| Component | Detail | Est. Cost |
|---|---|---|
| Serverless compute | Notebook execution (~20 min) | ~$0.50 |
| LLM tokens | 3 baseline queries + 2 tagged runs | ~$0.75 |
| Vector Search | Retrieval calls during agent queries | ~$0.25 |
| MLflow tracing | Trace storage (minimal) | ~$0.00 |
| **Total** | | **~$1-2** |